# PhishScamSense — Phishing URL Detection
End-to-end notebook: data loading → feature engineering → XGBoost training → evaluation → inference demo

## 1. Setup & Imports

In [ ]:
# Install dependencies if needed
# !pip install xgboost tldextract scikit-learn beautifulsoup4 python-whois requests

import sys
from pathlib import Path

# Add project root to path
PROJECT_ROOT = Path("../..").resolve()
sys.path.insert(0, str(PROJECT_ROOT))

import numpy as np
import pandas as pd
import xgboost as xgb
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay

from ml.src.data.data_loader import load_cic_bell_dns2021
from ml.src.features.feature_extractor import extract_features, FEATURE_COUNT
from ml.src.features.url_features import extract_url_features
from ml.src.features.whitelist import is_whitelisted

CLASS_NAMES = ["benign", "phishing", "malware", "spam"]
print(f"Feature count: {FEATURE_COUNT}")

## 2. Load Dataset

In [ ]:
DATA_DIR = PROJECT_ROOT / "data" / "raw"

urls, labels = load_cic_bell_dns2021(
    data_dir=DATA_DIR,
    max_benign=100_000,
    seed=42,
)

df = pd.DataFrame({"url": urls, "label": labels})
df["class"] = df["label"].map(dict(enumerate(CLASS_NAMES)))

print(f"Total samples: {len(df):,}")
print(df["class"].value_counts())

## 3. Feature Engineering

In [ ]:
# Example feature extraction for a single URL
sample_urls = [
    "https://google.com",
    "https://paypal-secure-verify.xyz/confirm?token=abc",
    "https://sso.ui.ac.id/cas/login",
]

for url in sample_urls:
    wl = is_whitelisted(url)
    feats = extract_url_features(url)
    print(f"\nURL: {url}")
    print(f"  Whitelisted: {wl}")
    print(f"  domain_in_brand_exact: {feats['domain_in_brand_exact']}")
    print(f"  typosquatting_min_dist: {feats['typosquatting_min_dist']}")
    print(f"  has_suspicious_tld: {feats['has_suspicious_tld']}")
    print(f"  phish_hints_count: {feats['phish_hints_count']}")
    print(f"  has_https: {feats['has_https']}")

## 4. Batch Feature Extraction

In [ ]:
import logging
logging.basicConfig(level=logging.INFO, format="%(asctime)s %(message)s", datefmt="%H:%M:%S")

# Use a balanced sample for notebook runs (full training uses train.py)
SAMPLES_PER_CLASS = 4000

from collections import defaultdict
import random

rng = random.Random(42)
sampled = defaultdict(list)
for url, label in zip(urls, labels):
    sampled[label].append(url)

sample_urls, sample_labels = [], []
for label in range(4):
    pool = sampled[label]
    n = min(SAMPLES_PER_CLASS, len(pool))
    chosen = rng.sample(pool, n)
    sample_urls.extend(chosen)
    sample_labels.extend([label] * n)

print(f"Sampled: {len(sample_urls):,} URLs")

# Extract features
rows = []
for url in sample_urls:
    try:
        feat = extract_features(url, html=None, compute_external=False)
        rows.append(list(feat.values()))
    except Exception:
        rows.append([0.0] * FEATURE_COUNT)

X = np.array(rows, dtype=np.float32)
y = np.array(sample_labels, dtype=np.int32)
print(f"Feature matrix shape: {X.shape}")

## 5. Train/Val/Test Split

In [ ]:
X_train_val, X_test, y_train_val, y_test = train_test_split(X, y, test_size=0.15, stratify=y, random_state=42)
X_train, X_val, y_train, y_val = train_test_split(X_train_val, y_train_val, test_size=0.15, stratify=y_train_val, random_state=42)

print(f"Train: {X_train.shape[0]:,}  Val: {X_val.shape[0]:,}  Test: {X_test.shape[0]:,}")

## 6. XGBoost Training

In [ ]:
dtrain = xgb.DMatrix(X_train, label=y_train)
dval   = xgb.DMatrix(X_val,   label=y_val)
dtest  = xgb.DMatrix(X_test,  label=y_test)

params = {
    "objective":        "multi:softmax",
    "num_class":        4,
    "eval_metric":      "mlogloss",
    "max_depth":        8,
    "learning_rate":    0.1,
    "n_estimators":     500,
    "subsample":        0.8,
    "colsample_bytree": 0.8,
    "min_child_weight": 3,
    "use_label_encoder": False,
    "verbosity":        1,
    "seed":             42,
}

evals_result = {}
booster = xgb.train(
    params,
    dtrain,
    num_boost_round=500,
    evals=[(dval, "validation")],
    evals_result=evals_result,
    verbose_eval=50,
)
print("Training complete.")

## 7. Evaluation

In [ ]:
y_pred = booster.predict(dtest).astype(int)

print("Test-set Classification Report:")
print(classification_report(y_test, y_pred, target_names=CLASS_NAMES, digits=4))

In [ ]:
fig, ax = plt.subplots(figsize=(7, 6))
cm = confusion_matrix(y_test, y_pred)
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=CLASS_NAMES)
disp.plot(ax=ax, colorbar=False, cmap="Blues")
ax.set_title("Confusion Matrix — PhishScamSense")
plt.tight_layout()
plt.show()

In [ ]:
feat_names = list(extract_features("https://example.com", html=None, compute_external=False).keys())
scores = booster.get_score(importance_type="gain")

importance_df = pd.DataFrame([
    {"feature": k, "gain": v}
    for k, v in scores.items()
]).sort_values("gain", ascending=False).head(20)

fig, ax = plt.subplots(figsize=(9, 6))
sns.barplot(data=importance_df, x="gain", y="feature", palette="viridis", ax=ax)
ax.set_title("Top 20 Features by Gain")
ax.set_xlabel("Gain")
ax.set_ylabel("")
plt.tight_layout()
plt.show()

## 8. Learning Curve

In [ ]:
losses = evals_result["validation"]["mlogloss"]
plt.figure(figsize=(8, 4))
plt.plot(losses, label="Validation log-loss")
plt.xlabel("Boosting round")
plt.ylabel("mlogloss")
plt.title("XGBoost Learning Curve")
plt.legend()
plt.tight_layout()
plt.show()

## 9. Save Model

In [ ]:
OUTPUT_DIR = PROJECT_ROOT / "ml" / "exports"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

model_path = OUTPUT_DIR / "xgb_classifier.json"
booster.save_model(str(model_path))
print(f"Model saved to: {model_path}")

## 10. Inference Demo

In [ ]:
# Load model and run inference
loaded_booster = xgb.Booster()
loaded_booster.load_model(str(model_path))

demo_urls = [
    "https://www.google.com/search?q=phishing+detection",
    "https://sso.ui.ac.id/cas/login",
    "http://paypal-secure-update.xyz/verify?account=1234",
    "http://192.168.1.1/admin",
    "https://github.com/user/repo",
    "http://xn--pypal-4ve.com/login",  # punycode paypal typosquat
]

print(f"{'URL':<55} {'Whitelisted':<12} {'Prediction':<12} {'Confidence'}")
print("-" * 95)
for url in demo_urls:
    if is_whitelisted(url):
        print(f"{url:<55} {'YES':<12} {'benign':<12} 1.000")
        continue
    feats = extract_features(url, html=None, compute_external=False)
    feat_vals = np.array([list(feats.values())], dtype=np.float32)
    dm = xgb.DMatrix(feat_vals)
    margins = loaded_booster.predict(dm, output_margin=True)
    e = np.exp(margins - margins.max(axis=1, keepdims=True))
    proba = (e / e.sum(axis=1, keepdims=True))[0]
    pred = int(np.argmax(proba))
    print(f"{url:<55} {'NO':<12} {CLASS_NAMES[pred]:<12} {proba[pred]:.3f}")